In [2]:
import torch

class TinyModel(torch.nn.Module):
    def __init__(self):
        super(TinyModel, self).__init__()

        self.linear1 = torch.nn.Linear(100, 200)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(200, 10)
        self.softmax = torch.nn.Softmax()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.softmax(x)
        return x

tinymodel = TinyModel()

print('The model:')
print(tinymodel)

print('\n\nJust one layer:')
print(tinymodel.linear2)

print('\n\nModel params:')
for param in tinymodel.parameters():
    print(param)

print('\n\nLayer params:')
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
  (softmax): Softmax(dim=None)
)


Just one layer:
Linear(in_features=200, out_features=10, bias=True)


Model params:
Parameter containing:
tensor([[-0.0665,  0.0579,  0.0760,  ...,  0.0655,  0.0524, -0.0439],
        [-0.0967,  0.0917,  0.0858,  ...,  0.0524, -0.0501,  0.0010],
        [ 0.0873,  0.0692, -0.0142,  ...,  0.0625,  0.0029,  0.0675],
        ...,
        [ 0.0147, -0.0632, -0.0078,  ...,  0.0320,  0.0094,  0.0259],
        [-0.0186, -0.0180,  0.0953,  ...,  0.0773,  0.0247, -0.0819],
        [-0.0232,  0.0606, -0.0803,  ...,  0.0702,  0.0481, -0.0240]],
       requires_grad=True)
Parameter containing:
tensor([ 0.0695, -0.0482, -0.0384, -0.0140,  0.0108,  0.0316,  0.0024,  0.0001,
         0.0999,  0.0701, -0.0298, -0.0132,  0.0293,  0.0117,  0.0673,  0.0893,
        -0.0538,  0.0112, -0.0332,  0.07

In [4]:
# The most basic type of neural network layer is a linear or fully connected layer.
# In this type of nn, every input influences every output of the layer to a degree specified by a layer's weights.

lin = torch.nn.Linear(3,2)
x = torch.rand(1,3)
print('Input: ')
print(x)

print('\n\nWeight and bias parameters:')
for param in lin.parameters():
    print(param)

y = lin(x)
print('\n\nOutput:')
print(y)

Input: 
tensor([[0.9159, 0.4716, 0.5832]])


Weight and bias parameters:
Parameter containing:
tensor([[ 0.1694, -0.0712, -0.2570],
        [ 0.5022,  0.3036,  0.4735]], requires_grad=True)
Parameter containing:
tensor([-0.3117,  0.2846], requires_grad=True)


Output:
tensor([[-0.3399,  1.1639]], grad_fn=<AddmmBackward0>)


In [6]:
# Convolutional Layers are built to handle data with a high degree of spatial correlation.
import torch.nn.functional as F

class LeNet(torch.nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # 1 input image channel (black & white), 6 output channels, 5x5 square convolution kernel
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        self.conv2 = torch.nn.Conv2d(6, 16, 3)
        # an affine operation: y = Wx + b
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120) 
        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        # get all dimensions except batch number
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

In [9]:
"""
 In the code block above, LeNet takes in a 1x32x32 image that is black and white.
 The first argument to the convolutional network's constructor is the number of input channels. (in this case, 1).
 A convolutional layer is like a window scanning for patterns across the image, a pattern is called a feature.
 One of the parameters to a convolutional layer is the number of features we would like it to learn.
 The second argument to the constructor is the number of output features (which in this case is 6).
 The third argument is the window or kernel size (which in this case is 5 x 5). 
 If we want a window of two different numbers, we specify with a tuple.
"""

"""
Output of a convolutional layer is an activation map which is a spatial representation of the presence of features in the input tensor.
conv1 gives us an output tensor of 6x28x28 (6 features 28 is height and width)

"""

'\nOutput of a convolutional layer is an activation map which is a spatial representation of the presence of features in the input tensor.\nconv1 gives us an output tensor of 6x28x28 (6 features 28 is height and width)\n\n'

In [10]:
# Recurrent Neural Networks are used for sequential data (ie. time series measurements)
# And rnn does this by maintaining a hidden state that acts as a memory for what has been seen through the sequence so far.
class LSTMTagger(torch.nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim
        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)
        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim)
        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

In [11]:
"""
The constructor has four arguments:
    vocab_size: the number of words in the input vocabulary. Each word is a unit vector in a vocab_size dimensional space.
    tagset_size: the number of tags in the output set.
    embedding_dim: the size of the embedding space for the vocabulary. An embedding maps a vocabulary onto a low-dimensional space, where words
        with similar meanings are close together in space
    hidden_dim: is in the size of the LSTM's memory

The input will be a sentence with the words represented as indices of one-hot vectors.
The embedding layer will then map these down to an embedding_dim-dimensional space. 
The LSTM takes this sequence of embeddings and iterates over it, fielding an output vector of length hidden_dim. 
The final linear layer acts as a classifier; applying log_softmax() to the output of the final layer converts the output 
    into a normalized set of estimated probabilities that a given word maps to a given tag.
"""

'\nThe constructor has four arguments:\n'

In [ ]:
# There are also transformers, data manipulation layers, activation functions, and loss functions.